# 08. LoRA и QLoRA на синтетике (черновик, не запускался)

Одна сессия = одно обучение: модель (`MODEL`) и способ (`METHOD`: `qlora` — база в 4 битах, `lora` — база в bf16).
План и гипотезы — в `README.md`. Перед полным запуском поставь `N_TRAIN = 20`, `EPOCHS = 1` и проверь, что всё проходит.

Colab, A100. В `vkr1_data` на Drive кроме файлов этапа 07 нужны `train.jsonl` и (если сгенерирован) `train_names.jsonl`.

In [ ]:
%pip -q install -U accelerate bitsandbytes peft xgrammar

## Настройки

In [ ]:
import json, os, random, time, math
import numpy as np
import pandas as pd

MODELS = {"gemma4_e4b": "google/gemma-4-E4B-it", "phi4_mini": "microsoft/Phi-4-mini-instruct"}
MODEL = "gemma4_e4b"     # <-- gemma4_e4b (лучшая) или phi4_mini (самая слабая)
METHOD = "qlora"         # qlora: база в 4 битах (nf4); lora: база в bf16
TRAIN_FILES = ["train.jsonl", "train_names.jsonl"]   # второго файла может не быть
N_TRAIN = None           # сколько примеров взять; None = все (для проверки 20)
LR = 2e-4                # подбирается по real-val из {1e-4, 2e-4}
EPOCHS = 2
BATCH, ACCUM = 4, 4      # эффективный батч 16
MAX_LEN = 1024
LORA = dict(r=16, lora_alpha=32, lora_dropout=0.05)
SEED = 0

DATA, OUT = "data", "results"
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA, OUT = "/content/drive/MyDrive/vkr1_data", "/content/drive/MyDrive/vkr1_results"
except ImportError:
    pass
TAG = f"{MODEL}-{METHOD}"
ADAPTER = f"{OUT}/adapters/{TAG}_lr{LR:g}_ep{EPOCHS}"
os.makedirs(ADAPTER, exist_ok=True)
ATTRS = ["диагональ", "разрешение", "частота", "тип матрицы", "яркость", "контрастность", "изогнутость"]
random.seed(SEED); np.random.seed(SEED)

## Промпт (как в этапе 07, режим zero-shot)

In [ ]:
INSTRUCTION = """Извлеки из описания монитора значения семи характеристик и верни JSON.

Правила:
- Бери только то, что прямо написано в тексте описания. Если характеристики в тексте нет, пиши null.
- Не угадывай и не используй свои знания о модели монитора.
- Все значения строки в таком формате: диагональ — число в дюймах ("27"); разрешение — "2560x1440"; частота — число в герцах ("165"); тип матрицы — "IPS", "VA", "TN" или "OLED"; яркость — число в кд/м² ("300"); контрастность — число ("3000", а 1 млн к 1 пиши "1000000"); изогнутость — "да" или "нет".
- Ответ только JSON с ключами: диагональ, разрешение, частота, тип матрицы, яркость, контрастность, изогнутость."""

def make_prompt(text, mode):
    p = INSTRUCTION
    if mode == "few-shot":
        for i, ex in enumerate(shots, 1):
            p += f"\n\nПример {i}.\nОписание: {ex['description']}\nОтвет: {json.dumps(ex['gold'], ensure_ascii=False)}"
    return p + f"\n\nОписание: {text}\nОтвет:"

SCHEMA_JSON = {"type": "object",
               "properties": {a: {"type": ["string", "null"]} for a in ATTRS},
               "required": ATTRS, "additionalProperties": False}

## Обучающие примеры
Промпт zero-shot → ответ JSON эталона. Учится только ответ.

In [ ]:
rows = []
for f in TRAIN_FILES:
    p = f"{DATA}/{f}"
    if os.path.exists(p):
        rows += [json.loads(l) for l in open(p, encoding="utf-8")]
        print(f, "есть")
    else:
        print(f, "нет, пропускаю")
random.Random(SEED).shuffle(rows)
rows = rows[:N_TRAIN] if N_TRAIN else rows

def answer(gold):
    """ответ в том же виде, что требует инструкция: все семь ключей по порядку, null где нет"""
    return json.dumps({a: gold[a] for a in ATTRS}, ensure_ascii=False)

examples = [(make_prompt(r["description"], "zero-shot"), answer(r["gold"])) for r in rows]
print(len(examples), "примеров;", examples[0][1])

## Модель и адаптеры

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

name = MODELS[MODEL]
tok = AutoTokenizer.from_pretrained(name)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
kw = {"device_map": "auto"}
if METHOD == "qlora":
    kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16)
else:
    kw["torch_dtype"] = torch.bfloat16
torch.cuda.reset_peak_memory_stats()
try:
    model = AutoModelForCausalLM.from_pretrained(name, **kw)
except ValueError:                      # мультимодальные модели (Gemma 4) грузятся другим классом
    from transformers import AutoModelForImageTextToText
    model = AutoModelForImageTextToText.from_pretrained(name, **kw)
if METHOD == "qlora":
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
else:
    model.gradient_checkpointing_enable()
cfg = LoraConfig(task_type="CAUSAL_LM", target_modules="all-linear", **LORA)
model = get_peft_model(model, cfg)
model.print_trainable_parameters()
print("память после загрузки, ГБ:", round(torch.cuda.memory_allocated() / 1e9, 1))

In [ ]:
def encode(prompt, ans):
    """токены промпта (в шаблоне чата, как в generate) и ответа; метки -100 на промпте"""
    chat = [{"role": "user", "content": prompt}]
    p = tok.apply_chat_template(chat, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    p_ids = tok(p, add_special_tokens=False)["input_ids"]
    a_ids = tok(ans + (tok.eos_token or ""), add_special_tokens=False)["input_ids"]
    ids = (p_ids + a_ids)[:MAX_LEN]
    labels = ([-100] * len(p_ids) + a_ids)[:MAX_LEN]
    return ids, labels

data = [encode(p, a) for p, a in examples]
print("длина, токенов: медиана", int(np.median([len(i) for i, _ in data])), "максимум", max(len(i) for i, _ in data))

def batches(epoch):
    order = list(range(len(data)))
    random.Random(SEED + epoch).shuffle(order)
    for k in range(0, len(order), BATCH):
        part = [data[i] for i in order[k:k + BATCH]]
        L = max(len(i) for i, _ in part)
        ids = torch.full((len(part), L), tok.pad_token_id)
        lab = torch.full((len(part), L), -100)
        att = torch.zeros((len(part), L), dtype=torch.long)
        for j, (i, l) in enumerate(part):
            ids[j, :len(i)] = torch.tensor(i); lab[j, :len(l)] = torch.tensor(l); att[j, :len(i)] = 1
        yield ids.to(model.device), att.to(model.device), lab.to(model.device)

## Обучение
AdamW, линейный спад learning rate после 5% разогрева. Пиковая память обучения — для H7.

In [ ]:
from transformers import get_linear_schedule_with_warmup
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
steps = EPOCHS * math.ceil(len(data) / BATCH / ACCUM)
sched = get_linear_schedule_with_warmup(opt, int(0.05 * steps), steps)
torch.cuda.reset_peak_memory_stats()
model.train()
log, t0, step = [], time.time(), 0
for ep in range(EPOCHS):
    for k, (ids, att, lab) in enumerate(batches(ep)):
        loss = model(input_ids=ids, attention_mask=att, labels=lab).loss / ACCUM
        loss.backward()
        if (k + 1) % ACCUM == 0:
            opt.step(); sched.step(); opt.zero_grad(); step += 1
            log.append({"эпоха": ep, "шаг": step, "loss": loss.item() * ACCUM})
            if step % 10 == 0:
                print(ep, step, "/", steps, "loss", round(loss.item() * ACCUM, 4))
train = {"model": MODEL, "method": METHOD, "lr": LR, "epochs": EPOCHS, "n_train": len(data),
         "сек_обучения": round(time.time() - t0), "память_пик_обучения_ГБ": round(torch.cuda.max_memory_allocated() / 1e9, 1),
         "обучаемых_параметров": sum(p.numel() for p in model.parameters() if p.requires_grad)}
print(train)
model.save_pretrained(ADAPTER)
json.dump({"train": train, "log": log, "lora": LORA}, open(f"{ADAPTER}/train_log.json", "w"), ensure_ascii=False, indent=1)

## Оценка на real-val и heldout
Тот же протокол, что в 07: жадная генерация, 200 токенов, остановка на «}», zero-shot без схемы (H8) и со схемой. Метрики считает тот же код, что в 07.

In [ ]:
def parse(raw):
    """словарь по семи атрибутам или None, если ответ не разобрался как JSON"""
    m = re.search(r"\{.*\}", raw, re.S)
    try:
        d = json.loads(m.group()) if m else None
    except json.JSONDecodeError:
        d = None
    if not isinstance(d, dict):
        return None
    return {a: (None if d.get(a) in (None, "", "null") else str(d[a]).strip()) for a in ATTRS}

def number(s):
    return re.search(r"\d+(?:[.,]\d+)?", s).group().replace(",", ".")

def resolution(s):
    w, h = re.findall(r"\d+", s)[:2]
    return f"{w}x{h}" if int(w) >= 640 else None

def contrast(s):
    s = re.sub(r"(?<=\d)\s(?=\d{3}\b)", "", s.lower())
    v = float(number(s)) * (10**6 if re.search(r"m|м", s) else 1)
    return str(int(v)) if v >= 100 else None

NORM = {"диагональ": lambda s: f"{float(number(s)):g}", "разрешение": resolution, "частота": number,
        "тип матрицы": lambda s: s.upper().replace("QD-", ""), "яркость": number, "контрастность": contrast,
        "изогнутость": lambda s: s.lower()}

def norm(attr, v):
    try:
        return NORM[attr](v)
    except Exception:
        return v

def f1(tp, fp, fn):
    return 2 * tp / (2 * tp + fp + fn) if tp + fp + fn else 0.0

def score(rows, soft):
    cnt = {a: [0, 0, 0] for a in ATTRS}          # tp, fp, fn по атрибутам
    false_null = []                              # для пар с эталоном null: 1, если модель назвала значение
    exact = []                                   # все семь атрибутов верны
    for r in rows:
        pred = r["pred"] or {a: None for a in ATTRS}
        ok = True
        for a in ATTRS:
            g, p = r["gold"][a], pred[a]
            if soft and p is not None:
                p = norm(a, p)
            if g is None:
                false_null.append(p is not None)
                if p is not None:
                    cnt[a][1] += 1; ok = False
            elif p == g:
                cnt[a][0] += 1
            else:
                cnt[a][2] += 1; ok = False
                if p is not None:
                    cnt[a][1] += 1
        exact.append(ok)
    tp, fp, fn = (sum(c[i] for c in cnt.values()) for i in range(3))
    macro = np.mean([f1(*c) for c in cnt.values() if sum(c)])
    return {"micro_f1": f1(tp, fp, fn), "macro_f1": macro, "exact": np.mean(exact),
            "false_null": np.mean(false_null), "per_attr": {a: round(f1(*c), 3) for a, c in cnt.items()}}

def make_summary(rows, split, mode, schema):
    strict, soft = score(rows, False), score(rows, True)
    sec = [r["sec"] for r in rows]
    summary = {"split": split, "model": MODEL, "mode": mode, "schema": schema, "n": len(rows),
               "hf_id": MODELS[MODEL], "k_shot": K_SHOT if mode == "few-shot" else 0, "дата": time.strftime("%Y-%m-%d %H:%M"),
               "micro_f1_строго": round(strict["micro_f1"], 3), "micro_f1_норм": round(soft["micro_f1"], 3),
               "macro_f1_норм": round(soft["macro_f1"], 3), "exact_норм": round(soft["exact"], 3),
               "ложные_на_null": round(soft["false_null"], 3),
               "невалидных": round(np.mean([r["pred"] is None for r in rows]), 3),
               "сек_среднее": round(np.mean(sec), 2), "сек_p95": round(np.percentile(sec, 95), 2),
               "токенов_среднее": round(np.mean([r["tokens"] for r in rows]), 1),
               "токенов_вход_среднее": round(np.mean([r["tokens_in"] for r in rows if "tokens_in" in r] or [np.nan]), 1)}
    try:
        import torch, transformers
        summary["память_пик_ГБ"] = round(torch.cuda.max_memory_allocated() / 1e9, 1)
        summary["transformers"] = transformers.__version__
        summary["gpu"] = torch.cuda.get_device_name(0)       # время и память сравнимы только на одной и той же карте
        from importlib.metadata import version, PackageNotFoundError
        for pkg in ("xgrammar", "flash-linear-attention", "bitsandbytes"):
            try:
                summary[pkg.replace("-", "_")] = version(pkg)       # версии библиотек для журнала параметров
            except PackageNotFoundError:
                summary[pkg.replace("-", "_")] = "нет"
    except Exception:
        pass
    return summary

def save_summary(summary):
    """строка в общий файл (старая строка с тем же набором, моделью и режимом заменяется)"""
    sp = f"{OUT}/summary.csv"
    old = pd.read_csv(sp) if os.path.exists(sp) else pd.DataFrame()
    if len(old):
        same = ((old["split"] == summary["split"]) & (old["model"] == summary["model"])
                & (old["mode"] == summary["mode"]) & (old["schema"] == summary["schema"]))
        old = old[~same]
    pd.concat([old, pd.DataFrame([summary])]).to_csv(sp, index=False)

In [ ]:
# дальше: загрузка real_val и heldout, generate() и запись <набор>__<TAG>__zero-shot__<free|schema>.jsonl
# — скопировать ячейки «Данные», generate и «Прогон» из 07_run_model.ipynb, заменив MODEL на TAG
# и model на дообученную (model.eval()). Результаты класть в 08_lora/results.